Before this notebook, one should run [`data_retrieval.ipynb`](./data_retrieval.ipynb) to fetch LMP data from the ERCOT API.

In [5]:
import json

import polars as pl
import polars.selectors as cs

from pathlib import Path

from esrpy import (
    BacktestConfig,
    BatteryConfig,
    ConfigurableNonOptimizationStrategy,
    ConfigurableOptimizationStrategy,
    PointInTimeData,
    analyze_backtest_results,
    load_lmps,
    plot_single_backtest,
    run_backtest,
)

DATA_DIR = Path.cwd() / "data"
INPUTS_DIR = DATA_DIR / "inputs"
OUTPUTS_DIR = DATA_DIR / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

indicative_lmps, actual_lmps = load_lmps(
    actual_csv_path=INPUTS_DIR / "2025-10-01_to_2025-10-08_RHESS2_ESS1_actual_lmps.csv",
    forecast_csv_path=INPUTS_DIR / "2025-10-01_to_2025-10-08_RHESS2_ESS1_indicative_lmps.csv",
)

pit_data = PointInTimeData(indicative_lmps, actual_lmps)

print(f"Loaded {len(actual_lmps)} actual LMPs")
print(f"Loaded {len(indicative_lmps)} indicative LMPs")

Loaded 2011 actual LMPs
Loaded 22121 indicative LMPs


In [6]:
battery_config = BatteryConfig()
backtest_config = BacktestConfig()

# (name, strategy_instance, use_perfect_foresight)
strategies = [
    ("simple_threshold", ConfigurableNonOptimizationStrategy(), False),
    ("battery_optimization", ConfigurableOptimizationStrategy(), False),
    ("perfect_foresight", ConfigurableOptimizationStrategy(), True),
]

print(f"Battery config: {battery_config.capacity_mwh} MWh, {battery_config.max_power_mw} MW")
print(f"Backtest start: {backtest_config.start_time}")


Battery config: 100.0 MWh, 100.0 MW
Backtest start: None


In [ ]:
# run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_id = "2025-10-01_to_2025-10-08_RHESS2_ESS1_default"
plot_subtitle = "2025-10-01 to 2025-10-08 | RHESS2_ESS1"
results = {}

run_dir = OUTPUTS_DIR / run_id
results_dir = run_dir / "results_5min"
plots_dir = run_dir / "plots"
summaries_dir = run_dir / "summaries"

for d in [results_dir, plots_dir, summaries_dir]:
    d.mkdir(parents=True, exist_ok=True)

for strategy_name, strategy, use_perfect_foresight in strategies:
    print(f"\n{'='*50}")
    print(f"Running strategy: {strategy_name}")
    print(f"{'='*50}")

    results_df = run_backtest(
        pit_data=pit_data,
        strategy=strategy,
        battery_config=battery_config,
        backtest_config=backtest_config,
        use_perfect_foresight=use_perfect_foresight,
    )

    results_path = results_dir / f"{strategy_name}.csv"
    (
        results_df
        .with_columns(cs.float().round(2))
        .with_columns(pl.col("interval_end_cpt").dt.replace_time_zone(None))
        .write_csv(results_path, datetime_format="%Y-%m-%d %H:%M:%S")
    )
    print(f"Results saved to {results_path.relative_to(run_dir)}")

    plot_path = plots_dir / f"{strategy_name}.png"
    plot_single_backtest(
        results_df,
        output_path=plot_path,
        title=strategy_name,
        subtitle=plot_subtitle
    )
    print(f"Plot saved to {plot_path.relative_to(run_dir)}")

    summary = analyze_backtest_results(results_df, battery_config)

    summary_path = summaries_dir / f"{strategy_name}.json"
    with open(summary_path, "w") as f:
        json.dump(summary.model_dump(), f, indent=2)
    print(f"Summary saved to {summary_path.relative_to(run_dir)}")

    results[strategy_name] = {
        "df": results_df,
        "summary": summary,
    }

    print(f"\nTotal PnL: ${summary.total_pnl:,.2f}")
    print(f"Total cycles: {summary.total_cycles:.2f}")
    print(f"Award rate: {summary.award_rate:.1%}")

print(f"\n{'='*50}")
print("All backtests complete!")
print(f"{'='*50}")



Running strategy: simple_threshold
Results saved to results_5min/simple_threshold.csv
Plot saved to plots/simple_threshold.png
Summary saved to summaries/simple_threshold.json

Total PnL: $17,510.58
Total cycles: 2.77
Award rate: 3.0%

Running strategy: battery_optimization
Results saved to results_5min/battery_optimization.csv
Plot saved to plots/battery_optimization.png
Summary saved to summaries/battery_optimization.json

Total PnL: $18,430.13
Total cycles: 5.91
Award rate: 11.0%

Running strategy: perfect_foresight
Results saved to results_5min/perfect_foresight.csv
Plot saved to plots/perfect_foresight.png
Summary saved to summaries/perfect_foresight.json

Total PnL: $31,540.00
Total cycles: 10.36
Award rate: 23.0%

All backtests complete!


In [8]:
comparison = pl.DataFrame([
    {
        "strategy": name,
        "total_pnl": round(r["summary"].total_pnl, 0),
        "total_cycles": round(r["summary"].total_cycles, 0),
        "avg_pnl_per_interval": round(r["summary"].avg_pnl_per_interval, 2),
        "award_rate": round(r["summary"].award_rate, 2),
        "discharge_revenue": round(r["summary"].discharge_revenue, 0),
        "charge_cost": round(r["summary"].charge_cost, 0),
    }
    for name, r in results.items()
])

comparison


strategy,total_pnl,total_cycles,avg_pnl_per_interval,award_rate,discharge_revenue,charge_cost
str,f64,f64,f64,f64,f64,f64
"""simple_threshold""",17511.0,3.0,8.71,0.03,18500.0,-990.0
"""battery_optimization""",18430.0,6.0,9.17,0.11,24437.0,-6007.0
"""perfect_foresight""",31540.0,10.0,15.69,0.23,40821.0,-9281.0


## Findings

- **`simple_threshold`** generated the lowest (but still positive) PnL with minimal cycling.

  - Its naive thresholds resulted in cautious trading behavior.

  - The low cycling (2.77 cycles) suggests thresholds may have been too conservative, leaving profitable opportunities on the table.

![simple_threshold](data/outputs/2025-10-01_to_2025-10-08_RHESS2_ESS1_default/plots/simple_threshold.png)

- **`battery_optimization`** performed respectably in terms of PnL and demonstrated moderate cycling.

  - The strategy achieved ~$18.4K PnL with ~6 cycles—a reasonable balance.

  - A limitation is that the optimizer only sees 11 intervals (~55 min) ahead, which may cause it to miss longer-term price patterns.

  - A longer-horizon forecast—perhaps 12 hours—could help the optimizer anticipate larger price swings and manage state-of-charge more strategically.

![battery_optimization](data/outputs/2025-10-01_to_2025-10-08_RHESS2_ESS1_default/plots/battery_optimization.png)

- **`perfect_foresight`** achieved the highest PnL (~$31.5K) by knowing future prices and aggressively trading around price spreads.

  - However, it cycled nearly twice as much as the `battery_optimization` strategy (~10 vs ~6 cycles).

  - This aggressive cycling would be penalized if hurdle/cycling costs were better modeled in the objective function.

  - The gap between perfect foresight and battery optimization (~$13K) represents the theoretical value of improved forecasting.

![perfect_foresight](data/outputs/2025-10-01_to_2025-10-08_RHESS2_ESS1_default/plots/perfect_foresight.png)
